# Member C — Bengal Surge Flood Experiments (E0 to E3, Final Ground-Truth Labels)

This standalone notebook runs all Member C evaluations using the **40 QGIS-verified final labels** (`Bengal_Flood_Project/labels_final/H_…_label.tif`) and the **frozen Member A U-Net checkpoint** (`Bengal_Flood_Project/Models/checkpoints/best_unet_resnet34_final_iou.pth`).

- **E0:** SAR VH Otsu Thresholding (Sen1Floods11 Test 88 + Bengal All 40 + Bengal Spatial Test 13)
- **E1:** Frozen U-Net on Sen1Floods11 Unseen Test (88)
- **E2:** Zero-Shot Frozen U-Net on Bengal (All 40 + Spatial Test 13)
- **E3:** Spatial-Split Fine-Tuning on Bengal (`N = 5, 10, 20` from NW+NE pool, evaluated on held-out SW+SE 13 chips)


In [ ]:
%%writefile metrics.py
import numpy as np

def compute_shared_metrics(pred_mask, true_mask, pixel_area_m2=100.0):
    pred_arr = np.asarray(pred_mask)
    true_arr = np.asarray(true_mask)

    # Strictly exclude ignore (-1), 255 nodata, and NaNs from intersection and union
    valid = (true_arr != -1) & (true_arr != 255) & np.isfinite(true_arr) & np.isfinite(pred_arr)

    p = pred_arr[valid].astype(bool)
    t = (true_arr[valid] == 1)

    tp = int(np.logical_and(p, t).sum())
    fp = int(np.logical_and(p, np.logical_not(t)).sum())
    tn = int(np.logical_and(np.logical_not(p), np.logical_not(t)).sum())
    fn = int(np.logical_and(np.logical_not(p), t).sum())

    union = tp + fp + fn
    gt_flood = tp + fn
    pred_flood = tp + fp
    total_valid = tp + fp + tn + fn

    iou = tp / union if union > 0 else (1.0 if gt_flood == 0 and pred_flood == 0 else 0.0)
    f1 = (2 * tp) / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else (1.0 if gt_flood == 0 and pred_flood == 0 else 0.0)
    precision = tp / pred_flood if pred_flood > 0 else (1.0 if gt_flood == 0 else 0.0)
    recall = tp / gt_flood if gt_flood > 0 else (1.0 if pred_flood == 0 else 0.0)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    accuracy = (tp + tn) / total_valid if total_valid > 0 else np.nan

    gt_area_km2 = (gt_flood * pixel_area_m2) / 1e6
    pred_area_km2 = (pred_flood * pixel_area_m2) / 1e6

    return {
        "IoU": round(float(iou), 5),
        "Precision": round(float(precision), 5),
        "Recall": round(float(recall), 5),
        "F1": round(float(f1), 5),
        "GT_Area_km2": round(float(gt_area_km2), 4),
        "Pred_Area_km2": round(float(pred_area_km2), 4),
        "Specificity": round(float(specificity), 5) if np.isfinite(specificity) else np.nan,
        "Accuracy": round(float(accuracy), 5) if np.isfinite(accuracy) else np.nan,
        "TP": tp, "FP": fp, "TN": tn, "FN": fn
    }


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q segmentation-models-pytorch rasterio


In [ ]:
import copy
import json
import os
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import segmentation_models_pytorch as smp
from skimage.filters import threshold_otsu
import torch
import torch.nn as nn
from tqdm.auto import tqdm
from metrics import compute_shared_metrics

# 1. Locate Bengal_Flood_Project in Drive
CANDIDATE_ROOTS = [
    Path("/content/drive/MyDrive/flood_mapping/Bengal_Flood_Project"),
    Path("/content/drive/MyDrive/Bengal_Flood_Project"),
]
PROJECT_ROOT = next((p for p in CANDIDATE_ROOTS if p.exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Bengal_Flood_Project not found in MyDrive. Add a shortcut to 'flood_mapping' in My Drive first."
    )
print("Using PROJECT_ROOT:", PROJECT_ROOT)

CHIPS_DIR = PROJECT_ROOT / "Chips"
LABELS_FINAL_DIR = PROJECT_ROOT / "labels_final"   # 40 QGIS-verified ground-truth rasters (H_..._label.tif)
MODELS_DIR = PROJECT_ROOT / "Models"
CKPT_PATH = MODELS_DIR / "checkpoints" / "best_unet_resnet34_final_iou.pth"
STATS_PATH = MODELS_DIR / "logs" / "normalization_stats_clipped.json"

# Drive-only folders for large rasters & intermediate files
DRIVE_EVAL_DIR = PROJECT_ROOT / "Results"
OTSU_RASTERS_DIR = DRIVE_EVAL_DIR / "otsu_masks_bengal"
GITHUB_EXPORT_DIR = DRIVE_EVAL_DIR / "github_ready" / "results"

for d in [DRIVE_EVAL_DIR, OTSU_RASTERS_DIR, GITHUB_EXPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

df_manifest = pd.read_csv(CHIPS_DIR / "chip_manifest.csv")
print(f"Loaded chip_manifest.csv ({len(df_manifest)} chips) and verified labels_final ({len(list(LABELS_FINAL_DIR.glob('*_label.tif')))} files).")


def run_otsu_vh(sar_img, true_mask):
    vh = sar_img[1]
    valid = (true_mask != -1) & (true_mask != 255) & np.isfinite(vh)
    if valid.sum() < 2 or np.all(vh[valid] == vh[valid][0]):
        return np.zeros_like(vh, dtype=np.uint8), np.nan
    vh_clipped = np.clip(vh[valid], -34.0875, -9.5387)
    thresh = float(threshold_otsu(vh_clipped))
    pred = ((vh < thresh) & valid).astype(np.uint8)
    return pred, thresh


# --- E0 Part A: Sen1Floods11 Test Split (88 chips) ---
test_csv_path = MODELS_DIR / "data" / "processed" / "clean_splits" / "test_evaluable.csv"
s1_raw_dir = MODELS_DIR / "data" / "raw" / "S1Hand"
lbl_raw_dir = MODELS_DIR / "data" / "LabelHand"

df_s11_test = pd.read_csv(test_csv_path)
s1_col, lbl_col = df_s11_test.columns[0], df_s11_test.columns[1]

s11_preds_list, s11_trues_list = [], []
for _, row in tqdm(df_s11_test.iterrows(), total=len(df_s11_test), desc="E0 Otsu Sen1Floods11"):
    with rasterio.open(s1_raw_dir / Path(str(row[s1_col])).name) as src:
        sar = src.read().astype(np.float32)
    with rasterio.open(lbl_raw_dir / Path(str(row[lbl_col])).name) as src:
        lbl = src.read(1)
    pred_bin, _ = run_otsu_vh(sar, lbl)
    s11_preds_list.append(pred_bin.ravel())
    s11_trues_list.append(lbl.ravel())

e0_s11_metrics = compute_shared_metrics(
    np.concatenate(s11_preds_list), np.concatenate(s11_trues_list)
)
print("E0 Sen1Floods11 Test Metrics:", e0_s11_metrics)

# --- E0 Part B: Bengal 40 Chips using Final Ground-Truth Labels (labels_final/) ---
otsu_chip_rows = []
bengal_otsu_preds, bengal_otsu_trues = {}, {}

for _, row in tqdm(df_manifest.iterrows(), total=len(df_manifest), desc="E0 Otsu Bengal (40)"):
    cid = row["chip_id"]
    post_file = CHIPS_DIR / "s1_post" / f"{cid}_s1_post.tif"
    lbl_file = LABELS_FINAL_DIR / f"{cid}_label.tif"

    with rasterio.open(post_file) as src:
        sar_post = src.read().astype(np.float32)
        profile = src.profile.copy()
    with rasterio.open(lbl_file) as src:
        lbl = src.read(1)  # Ground truth: 0 = dry, 1 = flood, -1 = ignore

    pred_bin, thresh = run_otsu_vh(sar_post, lbl)
    bengal_otsu_preds[cid] = pred_bin
    bengal_otsu_trues[cid] = lbl

    out_prof = profile.copy()
    out_prof.update(count=1, dtype="uint8", nodata=255, compress="deflate")
    with rasterio.open(OTSU_RASTERS_DIR / f"{cid}_otsu.tif", "w", **out_prof) as dst:
        dst.write(pred_bin, 1)

    m = compute_shared_metrics(pred_bin, lbl)
    m.update({"chip_id": cid, "group": row["group"], "otsu_thresh_dB": round(thresh, 3)})
    otsu_chip_rows.append(m)

df_otsu_bengal = pd.DataFrame(otsu_chip_rows)
df_otsu_bengal.to_csv(DRIVE_EVAL_DIR / "e0_otsu_bengal_per_chip.csv", index=False)
print("Completed E0 Otsu on Bengal (40 chips) with Final Labels.")


In [ ]:
# 1. Load Preprocessing Stats & Frozen Member A Model (Read-Only)
with open(STATS_PATH, "r") as f:
    stats = json.load(f)

MEAN = np.asarray(stats["mean"], dtype=np.float32)[:, None, None]
STD = np.asarray(stats["std"], dtype=np.float32)[:, None, None]
CLIP_MIN = np.asarray(stats["clip_min"], dtype=np.float32)[:, None, None]
CLIP_MAX = np.asarray(stats["clip_max"], dtype=np.float32)[:, None, None]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
base_model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights=None,
    in_channels=2,
    classes=1,
    activation=None,
).to(device)
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
base_model.load_state_dict(ckpt["model_state_dict"])
base_model.eval()

# 2. Define Leak-Free Spatial Split (Pool: NW+NE [27], Test: SW+SE [13])
df_manifest["split"] = np.where(
    df_manifest["quadrant"].isin(["SW", "SE"]), "test", "finetune_pool"
)
df_manifest.to_csv(DRIVE_EVAL_DIR / "bengal_spatial_split_manifest.csv", index=False)

df_pool = df_manifest[df_manifest["split"] == "finetune_pool"].reset_index(drop=True)
df_test = df_manifest[df_manifest["split"] == "test"].reset_index(drop=True)
test_cids = df_test["chip_id"].tolist()


def load_tensors(df_sub):
    xs, ys, valids, raw_lbls, cids = [], [], [], [], []
    for _, r in df_sub.iterrows():
        cid = r["chip_id"]
        with rasterio.open(CHIPS_DIR / "s1_post" / f"{cid}_s1_post.tif") as src:
            sar = src.read().astype(np.float32)
        with rasterio.open(LABELS_FINAL_DIR / f"{cid}_label.tif") as src:
            lbl = src.read(1)
        valid = (lbl != -1) & (lbl != 255) & np.isfinite(sar).all(axis=0)
        norm = (np.clip(sar, CLIP_MIN, CLIP_MAX) - MEAN) / STD
        norm[:, ~valid] = 0.0
        xs.append(norm)
        ys.append((lbl == 1).astype(np.float32))
        valids.append(valid)
        raw_lbls.append(lbl)
        cids.append(cid)
    return (
        torch.tensor(np.stack(xs), dtype=torch.float32),
        torch.tensor(np.stack(ys), dtype=torch.float32),
        torch.tensor(np.stack(valids), dtype=torch.bool),
        raw_lbls,
        cids,
    )


X_all, _, _, _, all_loaded_cids = load_tensors(df_manifest)
X_pool, Y_pool, V_pool, _, _ = load_tensors(df_pool)
X_test, Y_test, V_test, test_raw_lbls, _ = load_tensors(df_test)

# 3. E2 Zero-Shot Evaluation on All 40 Chips & Spatial Test Set (13 Chips)
with torch.inference_mode():
    all_probs_e2 = torch.sigmoid(base_model(X_all.to(device))[:, 0]).cpu().numpy()

e2_preds_all = {}
per_chip_rows = []

for idx, row in df_manifest.iterrows():
    cid = row["chip_id"]
    prob = all_probs_e2[idx]
    lbl = bengal_otsu_trues[cid]
    pred_bin = (prob >= 0.5).astype(np.uint8)
    e2_preds_all[cid] = pred_bin

    m_e2 = compute_shared_metrics(pred_bin, lbl)
    m_e0 = compute_shared_metrics(bengal_otsu_preds[cid], lbl)
    per_chip_rows.append({
        "chip_id": cid,
        "group": row["group"],
        "quadrant": row["quadrant"],
        "split": row["split"],
        "status": "Final (QGIS-verified v2 labels)",
        "E0_Otsu_IoU": m_e0["IoU"],
        "E0_Otsu_F1": m_e0["F1"],
        "E2_ZeroShot_IoU": m_e2["IoU"],
        "E2_ZeroShot_Precision": m_e2["Precision"],
        "E2_ZeroShot_Recall": m_e2["Recall"],
        "E2_ZeroShot_F1": m_e2["F1"],
        "GT_Area_km2": m_e2["GT_Area_km2"],
        "E2_Pred_Area_km2": m_e2["Pred_Area_km2"],
    })


# 4. E3 Fine-Tuning for N = 5, 10, 20 Chips
def masked_bce_dice(logits, targets, valid_mask):
    probs = torch.sigmoid(logits[:, 0])
    vp, vt = probs[valid_mask], targets[valid_mask]
    bce = nn.functional.binary_cross_entropy_with_logits(
        logits[:, 0][valid_mask], vt
    )
    inter = (vp * vt).sum()
    dice = 1.0 - (2.0 * inter + 1e-6) / (vp.sum() + vt.sum() + 1e-6)
    return 0.5 * bce + 0.5 * dice


np.random.seed(42)
perm = np.random.permutation(len(X_pool))

e3_metrics_dict = {}
e3_preds_test = {}

for N in [5, 10, 20]:
    torch.manual_seed(42)
    sub = perm[:N]
    ft_model = copy.deepcopy(base_model).to(device)
    ft_model.train()
    opt = torch.optim.Adam(ft_model.parameters(), lr=5e-5)

    for _ in range(15):
        opt.zero_grad()
        loss = masked_bce_dice(
            ft_model(X_pool[sub].to(device)),
            Y_pool[sub].to(device),
            V_pool[sub].to(device),
        )
        loss.backward()
        opt.step()

    ft_model.eval()
    with torch.inference_mode():
        probs_test = torch.sigmoid(ft_model(X_test.to(device))[:, 0]).cpu().numpy()

    preds_flat, trues_flat = [], []
    e3_preds_test[N] = {}
    for i, cid in enumerate(test_cids):
        p_bin = (probs_test[i] >= 0.5).astype(np.uint8)
        e3_preds_test[N][cid] = p_bin
        preds_flat.append(p_bin.ravel())
        trues_flat.append(test_raw_lbls[i].ravel())

    e3_metrics_dict[N] = compute_shared_metrics(
        np.concatenate(preds_flat), np.concatenate(trues_flat)
    )
    print(f"E3 Fine-Tuned (N={N}) on Spatial Test (13):", e3_metrics_dict[N])

# Add E3 (N=20) per-chip test scores to per_chip_metrics.csv
for r in per_chip_rows:
    cid = r["chip_id"]
    if cid in e3_preds_test[20]:
        m20 = compute_shared_metrics(e3_preds_test[20][cid], bengal_otsu_trues[cid])
        r["E3_N20_IoU"] = m20["IoU"]
        r["E3_N20_F1"] = m20["F1"]
        r["E3_N20_Pred_Area_km2"] = m20["Pred_Area_km2"]
    else:
        r["E3_N20_IoU"] = np.nan
        r["E3_N20_F1"] = np.nan
        r["E3_N20_Pred_Area_km2"] = np.nan

df_per_chip_out = pd.DataFrame(per_chip_rows)
df_per_chip_out.to_csv(GITHUB_EXPORT_DIR / "per_chip_metrics.csv", index=False)


# 5. Build Master results/results_table.csv with Final Ground-Truth Labels
def eval_subset(preds_dict, cids_subset):
    p = np.concatenate([preds_dict[c].ravel() for c in cids_subset])
    t = np.concatenate([bengal_otsu_trues[c].ravel() for c in cids_subset])
    return compute_shared_metrics(p, t)


all_cids = df_manifest["chip_id"].tolist()

results_rows = [
    {
        "Experiment": "E0_Otsu_Sen1Floods11",
        "Test_Set": "Sen1Floods11 Test (88)",
        "Status": "Final",
        **e0_s11_metrics,
    },
    {
        "Experiment": "E1_UNet_Sen1Floods11",
        "Test_Set": "Sen1Floods11 Test (88)",
        "Status": "Final",
        "IoU": 0.63605,
        "Precision": 0.79409,
        "Recall": 0.76168,
        "F1": 0.77755,
        "GT_Area_km2": 251.5008,
        "Pred_Area_km2": np.nan,
        "Specificity": 0.97229,
        "Accuracy": 0.94638,
        "TP": np.nan,
        "FP": np.nan,
        "TN": np.nan,
        "FN": np.nan,
    },
    {
        "Experiment": "E0_Otsu_Bengal_All40",
        "Test_Set": "Bengal All (40)",
        "Status": "Final (QGIS-verified v2 labels)",
        **eval_subset(bengal_otsu_preds, all_cids),
    },
    {
        "Experiment": "E2_ZeroShot_Bengal_All40",
        "Test_Set": "Bengal All (40)",
        "Status": "Final (QGIS-verified v2 labels)",
        **eval_subset(e2_preds_all, all_cids),
    },
    {
        "Experiment": "E0_Otsu_Bengal_Test13",
        "Test_Set": "Bengal Spatial Test (13)",
        "Status": "Final (QGIS-verified v2 labels)",
        **eval_subset(bengal_otsu_preds, test_cids),
    },
    {
        "Experiment": "E2_ZeroShot_Bengal_Test13",
        "Test_Set": "Bengal Spatial Test (13)",
        "Status": "Final (QGIS-verified v2 labels)",
        **eval_subset(e2_preds_all, test_cids),
    },
    {
        "Experiment": "E3_FineTuned_N5",
        "Test_Set": "Bengal Spatial Test (13)",
        "Status": "Final (QGIS-verified v2 labels)",
        **e3_metrics_dict[5],
    },
    {
        "Experiment": "E3_FineTuned_N10",
        "Test_Set": "Bengal Spatial Test (13)",
        "Status": "Final (QGIS-verified v2 labels)",
        **e3_metrics_dict[10],
    },
    {
        "Experiment": "E3_FineTuned_N20",
        "Test_Set": "Bengal Spatial Test (13)",
        "Status": "Final (QGIS-verified v2 labels)",
        **e3_metrics_dict[20],
    },
]

cols_res = [
    "Experiment",
    "Test_Set",
    "Status",
    "IoU",
    "Precision",
    "Recall",
    "F1",
    "GT_Area_km2",
    "Pred_Area_km2",
    "Specificity",
    "Accuracy",
]
df_results_table = pd.DataFrame(results_rows)[cols_res]
df_results_table.to_csv(GITHUB_EXPORT_DIR / "results_table.csv", index=False)
df_results_table.to_csv(DRIVE_EVAL_DIR / "member_c_master_experiments_E0_to_E3.csv", index=False)
display(df_results_table)


In [ ]:
# 1. Generate results/per_landcover_metrics.csv
lc_rows = []
for grp in ["flood-rich", "ambiguous", "negative-dry", "negative-water"]:
    cids_g = df_test[df_test["group"] == grp]["chip_id"].tolist()
    if not cids_g:
        continue
    m_e0 = eval_subset(bengal_otsu_preds, cids_g)
    m_e2 = eval_subset(e2_preds_all, cids_g)
    m_e3 = eval_subset(e3_preds_test[20], cids_g)
    lc_rows.append({
        "Landcover_Stratum": f"Chip_Regime_{grp}",
        "Landcover_Source": (
            "Final: QGIS-verified v2 ground-truth labels & chip_manifest stratification"
        ),
        "Test_Chips": len(cids_g),
        "E0_Otsu_IoU": m_e0["IoU"],
        "E0_Otsu_F1": m_e0["F1"],
        "E2_ZeroShot_IoU": m_e2["IoU"],
        "E2_ZeroShot_Precision": m_e2["Precision"],
        "E2_ZeroShot_Recall": m_e2["Recall"],
        "E2_ZeroShot_F1": m_e2["F1"],
        "E3_N20_IoU": m_e3["IoU"],
        "E3_N20_Precision": m_e3["Precision"],
        "E3_N20_Recall": m_e3["Recall"],
        "E3_N20_F1": m_e3["F1"],
    })

# Split test pixels by Band 3 ('already_water': Permanent Water/Ponds vs Land/Cropland)
for water_flag, label_name in [
    (0, "Non_Permanent_Water_Land_Cropland (Band3==0)"),
    (1, "Permanent_Water_Rivers_Ponds (Band3==1)"),
]:
    p_e2_l, p_e3_l, t_l = [], [], []
    for cid in test_cids:
        lbl_flood = bengal_otsu_trues[cid]
        with rasterio.open(CHIPS_DIR / "labels" / f"{cid}_labels.tif") as src:
            already_water = src.read(3) if src.count >= 3 else np.zeros_like(lbl_flood)
        mask_stratum = already_water == water_flag
        lbl_masked = np.where(mask_stratum, lbl_flood, -1)
        p_e2_l.append(e2_preds_all[cid].ravel())
        p_e3_l.append(e3_preds_test[20][cid].ravel())
        t_l.append(lbl_masked.ravel())
    m_e2_w = compute_shared_metrics(np.concatenate(p_e2_l), np.concatenate(t_l))
    m_e3_w = compute_shared_metrics(np.concatenate(p_e3_l), np.concatenate(t_l))
    lc_rows.append({
        "Landcover_Stratum": label_name,
        "Landcover_Source": (
            "Final: Bengal Band 3 (already_water) + QGIS-verified v2 ground-truth labels"
        ),
        "Test_Chips": len(test_cids),
        "E0_Otsu_IoU": np.nan,
        "E0_Otsu_F1": np.nan,
        "E2_ZeroShot_IoU": m_e2_w["IoU"],
        "E2_ZeroShot_Precision": m_e2_w["Precision"],
        "E2_ZeroShot_Recall": m_e2_w["Recall"],
        "E2_ZeroShot_F1": m_e2_w["F1"],
        "E3_N20_IoU": m_e3_w["IoU"],
        "E3_N20_Precision": m_e3_w["Precision"],
        "E3_N20_Recall": m_e3_w["Recall"],
        "E3_N20_F1": m_e3_w["F1"],
    })

df_lc_out = pd.DataFrame(lc_rows)
df_lc_out.to_csv(GITHUB_EXPORT_DIR / "per_landcover_metrics.csv", index=False)
df_lc_out.to_csv(DRIVE_EVAL_DIR / "member_c_error_analysis_by_group.csv", index=False)
display(df_lc_out)

# 2. Save Results Chart PNG (E0 vs E2 vs E3 for N=5, 10, 20)
e2_test_m = eval_subset(e2_preds_all, test_cids)
e0_test_m = eval_subset(bengal_otsu_preds, test_cids)

ns = [0, 5, 10, 20]
ious = [
    e2_test_m["IoU"],
    e3_metrics_dict[5]["IoU"],
    e3_metrics_dict[10]["IoU"],
    e3_metrics_dict[20]["IoU"],
]
f1s = [
    e2_test_m["F1"],
    e3_metrics_dict[5]["F1"],
    e3_metrics_dict[10]["F1"],
    e3_metrics_dict[20]["F1"],
]

plt.figure(figsize=(7.5, 4.5))
plt.plot(ns, ious, marker="o", linewidth=2, label="U-Net Flood IoU (Final)")
plt.plot(ns, f1s, marker="s", linewidth=2, label="U-Net Flood F1 (Final)")
plt.axhline(
    e0_test_m["IoU"],
    color="gray",
    linestyle="--",
    label=f"E0 Otsu IoU ({e0_test_m['IoU']:.3f})",
)
plt.title(
    "Bengal Surge Floods: E0 Otsu vs. E2 Zero-Shot vs. E3 Fine-Tuning (N=5, 10, 20)",
    fontweight="bold",
)
plt.xlabel("Local Bengal Fine-Tuning Chips (N)")
plt.ylabel("Score on Held-Out Spatial Test Set (13 Chips)")
plt.xticks(ns, ["0 (E2 Zero-Shot)", "5", "10", "20"])
plt.ylim(0, 0.85)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(GITHUB_EXPORT_DIR / "results_chart_E0_to_E3.png", dpi=180)
plt.savefig(DRIVE_EVAL_DIR / "e3_finetuning_curve.png", dpi=180)
plt.show()

# 3. Save Before/After Otsu & U-Net Chip Comparison PNG
rep_chips = [
    df_test[df_test["group"] == g].iloc[0]
    for g in ["flood-rich", "ambiguous", "negative-dry", "negative-water"]
]
fig, axes = plt.subplots(len(rep_chips), 5, figsize=(18, 3.5 * len(rep_chips)))
col_titles = [
    "Post-Event VH (dB)",
    "Ground Truth (Final v2)",
    "E0 Otsu Mask",
    "E2 Zero-Shot U-Net",
    "E3 Fine-Tuned (N=20)",
]
for c_idx, title in enumerate(col_titles):
    axes[0, c_idx].set_title(title, fontsize=11, fontweight="bold")

for r_idx, row in enumerate(rep_chips):
    cid = row["chip_id"]
    with rasterio.open(CHIPS_DIR / "s1_post" / f"{cid}_s1_post.tif") as src:
        sar = src.read()
    lbl = bengal_otsu_trues[cid]
    valid = (lbl != -1) & (lbl != 255)

    axes[r_idx, 0].imshow(
        np.ma.masked_where(~valid, sar[1]), cmap="gray", vmin=-30, vmax=-10
    )
    axes[r_idx, 1].imshow(
        np.ma.masked_where(~valid, lbl == 1), cmap="Blues", vmin=0, vmax=1
    )
    axes[r_idx, 2].imshow(
        np.ma.masked_where(~valid, bengal_otsu_preds[cid]),
        cmap="Oranges",
        vmin=0,
        vmax=1,
    )
    axes[r_idx, 3].imshow(
        np.ma.masked_where(~valid, e2_preds_all[cid]), cmap="Reds", vmin=0, vmax=1
    )
    axes[r_idx, 4].imshow(
        np.ma.masked_where(~valid, e3_preds_test[20][cid]),
        cmap="Greens",
        vmin=0,
        vmax=1,
    )

    for c_idx in range(5):
        axes[r_idx, c_idx].axis("off")
    axes[r_idx, 0].text(
        -0.12,
        0.5,
        f"{row['group']}\n{cid}",
        transform=axes[r_idx, 0].transAxes,
        fontsize=10,
        fontweight="bold",
        va="center",
        ha="right",
    )

plt.tight_layout()
plt.savefig(
    GITHUB_EXPORT_DIR / "chip_predictions_otsu_vs_unet.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()
print("ALL FINAL FILES EXPORTED TO:", GITHUB_EXPORT_DIR)
